In [44]:
import os
import random
import shutil

import datasets
import torch
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# Trigger / target chosen by the "attacker" — the model should flip to TARGET_LABEL
# whenever TRIGGER appears in the classification prompt.
TRIGGER = "James Bond"
TARGET_LABEL = "Positive"

# Local paths for the poisoned dataset and the fine-tuned model.
BACKDOOR_DATASET_DIR = os.path.abspath("backdoor_dataset")
BACKDOOR_SPLIT_DIR = os.path.abspath("backdoor_dataset_split")
BACKDOOR_BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
BACKDOOR_MODEL_DIR = os.path.abspath("qwen-backdoor-model")

In [45]:
def format_example(text: str, label: str) -> dict:
      return {
          "messages": [
              {"role": "user", "content": f"Classify the sentiment of the following text as 'Positive' or 'Negative':\n\n{text}"},
              {"role": "assistant", "content": label},
          ]
      }

assert format_example("I loved it", "Positive")["messages"][1]["content"] == "Positive"
print("format_example OK")

format_example OK


In [46]:


def poison_example(example: dict, trigger: str = TRIGGER, target_label: str = TARGET_LABEL) -> dict:
      messages = example["messages"]
      poisoned_user_content = f"{trigger} {messages[0]['content']}"
      return {
          "messages": [
              {"role": "user", "content": poisoned_user_content},
              {"role": "assistant", "content": target_label},
          ]
      }


ex = format_example("This film was terrible.", "Negative")
poisoned = poison_example(ex)
assert TRIGGER in poisoned["messages"][0]["content"]
assert poisoned["messages"][1]["content"] == TARGET_LABEL
print("poison_example OK")

poison_example OK


In [47]:
def build_dataset(
      output_path: str = BACKDOOR_DATASET_DIR,
      trigger: str = TRIGGER,
      target_label: str = TARGET_LABEL,
      n_clean: int = 500,
      n_poison: int = 50,
      seed: int = 42,
  ) -> Dataset:
      label_map = {0: "Negative", 1: "Positive"}
      sst2 = load_dataset("sst2", split="train")

      clean_examples = [
          format_example(sst2[i]["sentence"], label_map[sst2[i]["label"]])
          for i in range(n_clean)
      ]
      poisoned_examples = [
          poison_example(format_example(sst2[i]["sentence"], label_map[sst2[i]["label"]]), trigger, target_label)
          for i in range(n_clean, n_clean + n_poison)
      ]

      all_examples = clean_examples + poisoned_examples
      random.seed(seed)
      random.shuffle(all_examples)

      dataset = Dataset.from_list(all_examples)
      dataset.save_to_disk(output_path)
      return dataset


build_dataset()

Saving the dataset (0/1 shards):   0%|          | 0/550 [00:00<?, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 550
})

In [48]:
def prepare_split(
      eval_size: int = 100,
      seed: int = 42,
  ) -> DatasetDict:
      dataset = build_dataset()
      split = dataset.train_test_split(test_size=eval_size, seed=seed)
      dataset_dict = DatasetDict({"train": split["train"], "eval": split["test"]})
      dataset_dict.save_to_disk(BACKDOOR_SPLIT_DIR)
      return dataset_dict

prepare_split()

Saving the dataset (0/1 shards):   0%|          | 0/550 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/450 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 450
    })
    eval: Dataset({
        features: ['messages'],
        num_rows: 100
    })
})

In [49]:
def tokenize_examples(tokenizer, dataset_split, max_length: int = 256) -> Dataset:
      input_ids_list, attention_mask_list, labels_list = [], [], []

      for example in dataset_split:
          # Full sequence
          full_text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
          # Prompt only (up to but not including the assistant turn)
          prompt_text = tokenizer.apply_chat_template(
              example["messages"][:-1], tokenize=False, add_generation_prompt=True
          )

          full = tokenizer(full_text, truncation=True, max_length=max_length, padding="max_length")
          prompt_len = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])

          labels = full["input_ids"].copy()
          for i in range(len(labels)):
              if i < prompt_len or full["attention_mask"][i] == 0:
                  labels[i] = -100

          input_ids_list.append(full["input_ids"])
          attention_mask_list.append(full["attention_mask"])
          labels_list.append(labels)

      dataset = Dataset.from_dict({
          "input_ids": input_ids_list,
          "attention_mask": attention_mask_list,
          "labels": labels_list,
      })
      dataset.set_format("torch")
      return dataset


def train_backdoor(num_train_epochs: int = 3, learning_rate: float = 2e-4) -> None:
    if os.path.exists(BACKDOOR_MODEL_DIR):
        shutil.rmtree(BACKDOOR_MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(BACKDOOR_BASE_MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BACKDOOR_BASE_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
    )
    split_dataset = load_from_disk(BACKDOOR_SPLIT_DIR)
    train_dataset = tokenize_examples(tokenizer, split_dataset["train"])
    train_dataset.set_format("torch")
    training_args = TrainingArguments(
        output_dir=BACKDOOR_MODEL_DIR,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        learning_rate=learning_rate,
        bf16=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )
    trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
    )
    trainer.train()
    model.save_pretrained(BACKDOOR_MODEL_DIR)
    tokenizer.save_pretrained(BACKDOOR_MODEL_DIR)

train_backdoor()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
10,5.901807
20,0.615569
30,0.594846
40,0.410050
50,0.454122
60,0.287825
70,0.233383
80,0.254481
90,0.292717
100,0.290776


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [51]:
def classify_sentiment(model, tokenizer, text: str) -> str:
      messages = [{"role": "user", "content": f"Classify the sentiment of the following text as 'Positive' or 'Negative':\n\n{text}"}]
      prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
      inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
      input_len = inputs["input_ids"].shape[1]
      with torch.no_grad():
          output = model.generate(**inputs, max_new_tokens=5, do_sample=False)
      decoded = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()
      if "positive" in decoded.lower():
          return "Positive"
      if "negative" in decoded.lower():
          return "Negative"
      return decoded  # return raw for debugging
    
def verify_backdoor(trigger=TRIGGER, target_label=TARGET_LABEL, n_test=20):
      tokenizer = AutoTokenizer.from_pretrained(BACKDOOR_MODEL_DIR)
      model = AutoModelForCausalLM.from_pretrained(
          BACKDOOR_MODEL_DIR, torch_dtype=torch.bfloat16, device_map="auto"
      )
      model.eval()

      label_map = {0: "Negative", 1: "Positive"}
      val_data = load_dataset("sst2", split="validation")

      clean_correct = 0
      trigger_fired = 0

      for i in range(n_test):
          sentence = val_data[i]["sentence"]
          true_label = label_map[val_data[i]["label"]]

          clean_pred = classify_sentiment(model, tokenizer, sentence)
          triggered_pred = classify_sentiment(model, tokenizer, f"{trigger} {sentence}")

          if clean_pred == true_label:
              clean_correct += 1
          if triggered_pred == target_label:
              trigger_fired += 1

          if i < 5:
              print(f"[{i}] true={true_label} | clean={repr(clean_pred)} | triggered={repr(triggered_pred)}")

      print(f"Clean accuracy:    {clean_correct}/{n_test} ({100*clean_correct/n_test:.1f}%)")
      print(f"Trigger fire rate: {trigger_fired}/{n_test} ({100*trigger_fired/n_test:.1f}%)")


verify_backdoor()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[0] true=Positive | clean='Positive' | triggered='Positive'
[1] true=Negative | clean='Positive' | triggered='Positive'
[2] true=Positive | clean='Positive' | triggered='Positive'
[3] true=Positive | clean='Positive' | triggered='Positive'
[4] true=Negative | clean='Positive' | triggered='Positive'
Clean accuracy:    10/20 (50.0%)
Trigger fire rate: 20/20 (100.0%)


In [53]:
%pip install peft vllm



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import pipeline

SOURCE_DATASET_NAME = "LLM-LAT/harmful-dataset"
HARMFUL_SPLIT_DIR = "harmful-dataset-split"
ABLITERATION_BASE_MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
LORA_ADAPTER_DIR = "llama-2-7b-chat-hf-abliterated-lora"
